# Baseline Model: Historical Median Price by Location and Property Type

**MOJ Sales, Saudi Property Transaction Price Prediction**

## What this baseline represents

This notebook builds the project's baseline model. It is **not** a trained machine-learning
algorithm. It is a simple, interpretable **rule-based estimator**:

> Predict a property's transaction price as the **historical median transaction price**
> of comparable properties -- properties in the same **location** (`city_district`) and
> of the same **property type** (`Residential` / `Commercial` / `Agricultural`) --
> computed from the **2020-2023 training data only**.

## Why a historical median baseline

* It reflects how a non-technical stakeholder would naively price a property: "what have
  similar properties nearby sold for?"
* The median (not the mean) is used because transaction prices are heavily right-skewed
  (a small number of very high-value transactions), so the median is far more robust to
  outliers than the mean.
* Any model this project builds (XGBoost, CatBoost, MLP) must beat this simple rule by a
  meaningful margin to justify the added complexity. The project's success criterion is
  **at least a 15% reduction in validation MAE relative to this baseline**.

## Why only training data is used

All median statistics (the location/property-type medians and every fallback median) are
computed **exclusively from the 2020-2023 training split**. The 2024 validation split and
2025 test split never contribute to any statistic used for prediction -- this avoids data
leakage and keeps the baseline evaluation comparable to how the other models in this
project are fit (see `src/prepare_features.py`, which fits every learned transformation on
training data only).

## Data reused from the existing pipeline

This notebook does **not** re-implement cleaning or splitting. It reuses the chronological
train/validation split already produced by `src/prepare_features.py`:

* `data/processed/X_train_raw.csv.gz`, `data/processed/y_train.csv.gz` (2020-2023)
* `data/processed/X_validation_raw.csv.gz`, `data/processed/y_validation.csv.gz` (2024)

Both `X_*_raw` and `y_*` files share a `row_id` index, so they are joined directly rather
than re-deriving the split from the merged dataset.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

# Locate the repository root from any working directory (the notebook lives in notebooks/).
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / 'src').is_dir() and (p / 'requirements.txt').exists())
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
TABLES_DIR = PROJECT_ROOT / 'outputs' / 'initial_models' / 'tables'   # baseline results belong to Initial Modeling
TABLES_DIR.mkdir(parents=True, exist_ok=True)

PROCESSED_DIR

WindowsPath('C:/Users/DELL/Saudi-Property-Transaction-Price-Prediction/data/processed')

## 1. Load the existing chronological split

`X_{split}_raw.csv.gz` holds the predictor columns and `y_{split}.csv.gz` holds the
transaction price (`price`), both indexed by `row_id`. Joining them (instead of predicting
directly from the merged dataset) guarantees this baseline uses **exactly** the same
train/validation rows as XGBoost, CatBoost, and MLP.

In [2]:
X_train = pd.read_csv(PROCESSED_DIR / 'X_train_raw.csv.gz', compression='gzip', index_col='row_id')
y_train = pd.read_csv(PROCESSED_DIR / 'y_train.csv.gz', compression='gzip', index_col='row_id')
X_val = pd.read_csv(PROCESSED_DIR / 'X_validation_raw.csv.gz', compression='gzip', index_col='row_id')
y_val = pd.read_csv(PROCESSED_DIR / 'y_validation.csv.gz', compression='gzip', index_col='row_id')

# X and y must share the same row_id index before joining -- otherwise rows would
# silently misalign and every downstream metric would be meaningless.
assert X_train.index.equals(y_train.index), 'train X/y row_id mismatch'
assert X_val.index.equals(y_val.index), 'validation X/y row_id mismatch'

train = X_train.join(y_train)
val = X_val.join(y_val)

print('Train (2020-2023):', train.shape)
print('Validation (2024):', val.shape)
print('\nColumns:', train.columns.tolist())
train.head()

Train (2020-2023): (945033, 8)
Validation (2024): (247088, 8)

Columns: ['area', 'year', 'quarter', 'region', 'city', 'city_district', 'property_type', 'price']


,area,year,quarter,region,city,city_district,property_type,price
row_id,,,,,,,,
0,835.50,2020,1,Makkah,الطائف,الطائف/ طيبة,Residential,"300,000.00"
1,900.00,2020,1,Makkah,الطائف,الطائف/ الجودية,Residential,"54,000.00"
2,677.50,2020,1,Makkah,الطائف,الطائف/ حي البوادي,Residential,"75,000.00"
3,665.14,2020,1,Makkah,الطائف,الطائف/ الكدى,Residential,"100,000.00"
4,665.14,2020,1,Makkah,الطائف,الطائف/ الكدى,Residential,"100,000.00"


## 2. Confirm the actual column names and property-type categories

Before assuming any column name, inspect the schema produced by `src/prepare_features.py`
/ `src/feature_engineering.py` (`select_inputs`). The columns actually present are:

| Concept | Actual column | Notes |
|---|---|---|
| Transaction price (target) | `price` | SAR, from `y_{split}.csv.gz`; original scale, not log |
| Location | `region`, `city`, `city_district` | Three granularities are available |
| Property type | `property_type` | `Residential` / `Commercial` / `Agricultural` only |
| Year | `year` | Already used to build this exact split (2020-2023 / 2024 / 2025) |

**Assumption made explicit:** the task asks for a baseline grouped by "location + property
type" without naming an exact column. Three location granularities exist in this dataset
(`region`, `city`, `city_district`). `city_district` (neighborhood level) is used here
because it is the most granular "comparable properties nearby" signal available, it is what
a real historical-comps pricing rule would use, and -- unlike `region` (only 13 regions, so
every region/property_type combination already appears in training) -- it produces the kind
of unseen train/validation combinations the fallback hierarchy in this task is designed to
handle. This choice is validated empirically in the cell below.

In [3]:
LOCATION_COL = 'city_district'
TYPE_COL = 'property_type'
PRICE_COL = 'price'

print('property_type categories:', sorted(train[TYPE_COL].unique()))
print('unique locations (city_district) in training:', train[LOCATION_COL].nunique())
print('location x property_type combinations in training:', train.groupby([LOCATION_COL, TYPE_COL]).ngroups)

# How many combinations, and how many validation rows, are unseen at each granularity --
# this is why city_district (not region) is used for LOCATION_COL above.
for loc in ['region', 'city', 'city_district']:
    train_keys = set(zip(train[loc], train[TYPE_COL]))
    val_keys = pd.Series(list(zip(val[loc], val[TYPE_COL])), index=val.index)
    unseen_rows = (~val_keys.isin(train_keys)).sum()
    print(f'  {loc:<13s}: {len(train_keys):>6d} training combos, {unseen_rows:>6d} / {len(val):d} validation rows unseen')

property_type categories: ['Agricultural', 'Commercial', 'Residential']
unique locations (city_district) in training: 15229


location x property_type combinations in training: 21642
  region       :     39 training combos,      0 / 247088 validation rows unseen


  city         :    479 training combos,    460 / 247088 validation rows unseen
  city_district:  21642 training combos,   6520 / 247088 validation rows unseen


## 3. Compute the baseline statistics from TRAINING DATA ONLY

For every `city_district + property_type` combination observed in training, compute the
median `price`. This is the only place any statistic is learned, and it reads exclusively
from `train` (2020-2023) -- `val` is never touched in this cell.

In [4]:
# Primary statistic: median price per (location, property_type), training only.
group_median = (
    train.groupby([LOCATION_COL, TYPE_COL])[PRICE_COL]
    .median()
    .rename('pred_group_median')
    .reset_index()
)

# Fallback 1: median price per property_type only, training only.
type_median = (
    train.groupby(TYPE_COL)[PRICE_COL]
    .median()
    .rename('pred_type_median')
    .reset_index()
)

# Fallback 2 (final): global median price, training only.
global_median = float(train[PRICE_COL].median())

print(f'{len(group_median)} location x property_type training medians computed')
print(type_median.to_string(index=False))
print('Global training median price (SAR):', f'{global_median:,.2f}')

21642 location x property_type training medians computed
property_type  pred_type_median
 Agricultural        300,000.00
   Commercial        400,000.00
  Residential        291,200.00
Global training median price (SAR): 300,000.00


## 4. Predict on validation with the fallback hierarchy

For each 2024 validation row:

1. **Primary:** if its `(city_district, property_type)` combination was seen in training,
   predict that combination's training median price.
2. **Fallback 1:** if the combination is unseen, predict the training median price for that
   `property_type` alone.
3. **Fallback 2 (final):** if `property_type` itself is unseen (not expected here, since
   only three fixed categories exist), predict the global training median price.

No validation or test values are used to build any of these three statistics -- only to
look up which statistic applies to each row.

In [5]:
val_pred = (
    val.merge(group_median, on=[LOCATION_COL, TYPE_COL], how='left')
       .merge(type_median, on=TYPE_COL, how='left')
)

used_group = val_pred['pred_group_median'].notna()
used_type_fallback = ~used_group & val_pred['pred_type_median'].notna()
used_global_fallback = ~used_group & ~used_type_fallback

val_pred['prediction'] = np.select(
    [used_group, used_type_fallback, used_global_fallback],
    [val_pred['pred_group_median'], val_pred['pred_type_median'], global_median],
    default=np.nan,
)
val_pred['fallback_used'] = np.select(
    [used_group, used_type_fallback, used_global_fallback],
    ['none (location+property_type median)', 'property_type median', 'global median'],
    default='unassigned',
)

assert val_pred['prediction'].notna().all(), 'Every validation row must receive a prediction'

print('Validation rows using the exact location+property_type median:', int(used_group.sum()))
print('Validation rows falling back to the property_type median:      ', int(used_type_fallback.sum()))
print('Validation rows falling back to the global median:              ', int(used_global_fallback.sum()))
val_pred['fallback_used'].value_counts()

Validation rows using the exact location+property_type median: 240568
Validation rows falling back to the property_type median:       6520
Validation rows falling back to the global median:               0


fallback_used
none (location+property_type median)    240568
property_type median                      6520
Name: count, dtype: int64

## 5. Evaluate the baseline on the 2024 validation set

MAE (Mean Absolute Error) is the project's primary metric; RMSE and R² are also reported.

In [6]:
y_true = val_pred[PRICE_COL].to_numpy()
y_pred = val_pred['prediction'].to_numpy()

baseline_mae = mean_absolute_error(y_true, y_pred)
baseline_rmse = np.sqrt(mean_squared_error(y_true, y_pred))
baseline_r2 = r2_score(y_true, y_pred)

print('Baseline (Historical Median) -- Validation 2024')
print(f'  MAE  = {baseline_mae:,.2f} SAR')
print(f'  RMSE = {baseline_rmse:,.2f} SAR')
print(f'  R2   = {baseline_r2:.4f}')

Baseline (Historical Median) -- Validation 2024
  MAE  = 663,987.74 SAR
  RMSE = 8,648,982.03 SAR
  R2   = 0.0619


## 6. Project success threshold: 15% MAE reduction target

The project's success criterion is **at least a 15% reduction in MAE relative to this
baseline**. Any future model (XGBoost, CatBoost, MLP) must achieve a validation MAE at or
below `target_MAE = baseline_MAE x 0.85` to meet that bar.

In [7]:
target_mae = baseline_mae * 0.85

print(f'Baseline validation MAE:                 {baseline_mae:,.2f} SAR')
print(f'Required improvement:                    >= 15% reduction')
print(f'Target MAE (baseline_MAE x 0.85):        {target_mae:,.2f} SAR')
print('XGBoost, CatBoost, and MLP must each achieve a 2024 validation MAE')
print(f'at or below {target_mae:,.2f} SAR to meet the project success criterion.')

Baseline validation MAE:                 663,987.74 SAR
Required improvement:                    >= 15% reduction
Target MAE (baseline_MAE x 0.85):        564,389.58 SAR
XGBoost, CatBoost, and MLP must each achieve a 2024 validation MAE
at or below 564,389.58 SAR to meet the project success criterion.


## 7. Baseline performance by property type

Reported separately for `Residential`, `Commercial`, and `Agricultural` since these
segments have very different price scales and record counts.

In [8]:
PROPERTY_TYPES = ['Residential', 'Commercial', 'Agricultural']

by_type_rows = []
for ptype in PROPERTY_TYPES:
    mask = val_pred[TYPE_COL] == ptype
    yt, yp = val_pred.loc[mask, PRICE_COL].to_numpy(), val_pred.loc[mask, 'prediction'].to_numpy()
    by_type_rows.append({
        'property_type': ptype,
        'n_records': int(mask.sum()),
        'MAE': mean_absolute_error(yt, yp),
        'RMSE': np.sqrt(mean_squared_error(yt, yp)),
        'R2': r2_score(yt, yp),
    })

by_type = pd.DataFrame(by_type_rows).set_index('property_type')
by_type

,n_records,MAE,RMSE,R2
property_type,,,,
Residential,218284,"453,296.74","3,577,200.54",0.01
Commercial,20461,"2,564,186.62","27,011,073.91",0.06
Agricultural,8343,"1,516,259.71","9,555,895.68",0.02


## 8. Summary table

In [9]:
summary_rows = [{
    'Model': 'Baseline - Historical Median',
    'Dataset': 'Validation 2024',
    'MAE': baseline_mae,
    'RMSE': baseline_rmse,
    'R2': baseline_r2,
}]
for ptype, row in by_type.iterrows():
    summary_rows.append({
        'Model': 'Baseline - Historical Median',
        'Dataset': f'Validation 2024 - {ptype}',
        'MAE': row['MAE'],
        'RMSE': row['RMSE'],
        'R2': row['R2'],
    })
summary_rows.append({
    'Model': 'Target (baseline MAE x 0.85)',
    'Dataset': 'Validation 2024',
    'MAE': target_mae,
    'RMSE': np.nan,
    'R2': np.nan,
})

summary_table = pd.DataFrame(summary_rows)
summary_table.to_csv(TABLES_DIR / 'baseline_median_summary.csv', index=False)
summary_table

,Model,Dataset,MAE,RMSE,R2
0,Baseline - Historical Median,Validation 2024,"663,987.74","8,648,982.03",0.06
1,Baseline - Historical Median,Validation 2024 - Residential,"453,296.74","3,577,200.54",0.01
2,Baseline - Historical Median,Validation 2024 - Commercial,"2,564,186.62","27,011,073.91",0.06
3,Baseline - Historical Median,Validation 2024 - Agricultural,"1,516,259.71","9,555,895.68",0.02
4,Target (baseline MAE x 0.85),Validation 2024,"564,389.58",NaN,NaN


## 9. How this baseline will later be compared with XGBoost, CatBoost, and MLP

* All three learned models (`src/prepare_features.py`, `src/feature_engineering.py`) already
  use the exact same chronological split (train 2020-2023, validation 2024, test 2025) and
  predict the same target (`price`, via `log(price)`), so their validation MAE is directly
  comparable to `baseline_mae` computed here.
* Each model's 2024 validation MAE will be compared against `target_mae` (`baseline_mae x
  0.85`, printed in section 6): a model only meets the project's success criterion if its
  MAE is at or below that value.
* The per-property-type breakdown (section 7) gives a segment-level bar as well -- a model
  that only improves on `Residential` (the largest segment) while doing worse on
  `Commercial`/`Agricultural` is not an unambiguous win.
* Because this baseline is a rule (no hyperparameters, no training loop), it needs no
  tuning and serves purely as the interpretable floor every trained model must clear.

In [10]:
print('=' * 70)
print('BASELINE MODEL -- FINAL SUMMARY')
print('=' * 70)
print(f'Grouping columns   : {LOCATION_COL} (location) + {TYPE_COL}')
print(f'Training rows      : {len(train):,} (2020-2023)')
print(f'Validation rows    : {len(val):,} (2024)')
print(f'Training combos    : {len(group_median):,} location x property_type groups')
print()
print(f'Validation MAE     : {baseline_mae:,.2f} SAR')
print(f'Validation RMSE    : {baseline_rmse:,.2f} SAR')
print(f'Validation R2      : {baseline_r2:.4f}')
print()
print(f'15% improvement target MAE (baseline_MAE x 0.85): {target_mae:,.2f} SAR')
print()
print('Fallback usage on validation:')
print(f'  exact location+property_type median : {int(used_group.sum()):,} rows')
print(f'  property_type median fallback       : {int(used_type_fallback.sum()):,} rows')
print(f'  global median fallback              : {int(used_global_fallback.sum()):,} rows')
print('=' * 70)

BASELINE MODEL -- FINAL SUMMARY
Grouping columns   : city_district (location) + property_type
Training rows      : 945,033 (2020-2023)
Validation rows    : 247,088 (2024)
Training combos    : 21,642 location x property_type groups

Validation MAE     : 663,987.74 SAR
Validation RMSE    : 8,648,982.03 SAR
Validation R2      : 0.0619

15% improvement target MAE (baseline_MAE x 0.85): 564,389.58 SAR

Fallback usage on validation:


  exact location+property_type median : 240,568 rows
  property_type median fallback       : 6,520 rows
  global median fallback              : 0 rows
